<h1 style="color:#004080; font-size:2em; margin-bottom:0.3em;">
  Prototype: Interactive HUC Selection Map
</h1>
<h4 style="color:gray; font-weight:normal; line-height:1.5;">
  This is a <strong>prototype</strong> notebook testing an interactive way to build a HUC-based gage subset: choose a HUC level (HUC6 or HUC8), click a region on the map, and get back a cached table of just the gages inside that region. It builds on the lookup table produced by <code>huc-lookup.ipynb</code> — run that notebook first so <code>data/gage_huc_lookup.parquet</code> exists. Once this interaction design feels right, we'll decide how to merge it into <code>huc-lookup.ipynb</code>.
</h4>

<a id="authenticate"></a>
## Authenticate with HydroShare

In [1]:
from stream_utils import S3hsclient

hs = S3hsclient.S3HydroShare()

Please Enter Your HydroShare Credentials


## Load gage metadata and the cached HUC lookup table

We need `latitude_wgs84`/`longitude_wgs84` from `metadata_df` (to plot gages) and `HUC6`/`HUC8` from the cache built by `huc-lookup.ipynb`:

In [2]:
import os
import pandas

resource_id = '4553ef8a14884e4199c7018975b8faaa'
resource = hs.resource(resource_id)

metadata_df = pandas.read_parquet(
    f'tonycastronova/{resource_id}/data/contents/metadata.parquet',
    filesystem=hs.get_s3_filesystem()
)

huc_lookup_path = 'data/gage_huc_lookup.parquet'
if not os.path.exists(huc_lookup_path):
    raise FileNotFoundError(
        f"'{huc_lookup_path}' not found. Run huc-lookup.ipynb first to build "
        "the HUC lookup cache before using this notebook."
    )

huc_lookup = pandas.read_parquet(huc_lookup_path)

gages_df = metadata_df[['STREAM_ID', 'latitude_wgs84', 'longitude_wgs84']].merge(
    huc_lookup[['STREAM_ID', 'HUC6', 'HUC8']], on='STREAM_ID', how='inner'
)
gages_df.head()

,STREAM_ID,latitude_wgs84,longitude_wgs84,HUC6,HUC8
0,STREAM-gauge-1,31.306583,-89.264111,031700,03170005
1,STREAM-gauge-2,31.221667,-89.053056,031700,03170005
2,STREAM-gauge-3,31.025278,-89.016667,031700,03170007
3,STREAM-gauge-4,30.367778,-88.563056,031700,03170006
4,STREAM-gauge-5,30.382694,-88.608444,031700,03170006


## Choose a HUC level

Pick `HUC6` (coarser, fewer regions) or `HUC8` (finer, more regions). Re-run the cells below after changing this:

In [3]:
import ipywidgets as widgets

huc_level_widget = widgets.Dropdown(
    options=['HUC6', 'HUC8'],
    value='HUC8',
    description='HUC level:',
)
huc_level_widget

Dropdown(description='HUC level:', index=1, options=('HUC6', 'HUC8'), value='HUC8')

In [4]:
huc_level = huc_level_widget.value
huc_field = huc_level.lower()

print(f'Using {huc_level}')

Using HUC8


## Fetch boundary polygons for just the HUCs present in our gage set

Rather than fetching every HUC in the country, we only fetch boundaries for the HUC codes that actually appear among our gages — this keeps the request small and fast:

In [5]:
import geopandas as gpd
import pygeohydro as gh
from tqdm import tqdm


def fetch_huc_boundaries(huc_ids, huc_field, batch_size=50):
    """Fetch HUC boundary polygons for a list of HUC ids, in batches."""
    wbd = gh.WBD(huc_field)
    results = []

    batches = [huc_ids[i:i + batch_size] for i in range(0, len(huc_ids), batch_size)]
    for batch in tqdm(batches, desc=f'Fetching {huc_field} boundaries'):
        try:
            results.append(wbd.byids(huc_field, batch))
        except Exception as e:
            print(f'Batch failed ({len(batch)} {huc_field}s): {e}')

    return gpd.GeoDataFrame(pandas.concat(results, ignore_index=True), crs=results[0].crs)

In [6]:
huc_ids = gages_df[huc_level].dropna().unique().tolist()
print(f'{len(huc_ids)} unique {huc_level} regions among our gages')

boundaries_gdf = fetch_huc_boundaries(huc_ids, huc_field)
boundaries_gdf[[huc_field, 'name']].head()

964 unique HUC8 regions among our gages


Fetching huc8 boundaries:  20%|██        | 4/20 [00:19<01:20,  5.02s/it]/Users/igarousi/Projects/2026_STREAM/STREAM/.pixi/envs/default/lib/python3.14/site-packages/pygeoogc/core.py:293: UserWarning: Found 1 failed request. Retrying ...
  resp = self._cleanup_resp(resp, payloads)
Fetching huc8 boundaries: 100%|██████████| 20/20 [02:18<00:00,  6.91s/it]


,huc8,name
0,11100302,Lower North Canadian
1,11110103,Illinois
2,02070008,Middle Potomac-Catoctin
3,02070010,Middle Potomac-Anacostia-Occoquan
4,03010107,Lower Roanoke


## Build the interactive map

Click a HUC boundary to select it — the boundary highlights, a popup shows its name and gage count, and the matching subset of gages is stored in `selection['gages_df']` for the save step below.

In [7]:
from stream_utils import huc_map_selector

m, selection = huc_map_selector.build_huc_selection_map(gages_df, boundaries_gdf, huc_level, huc_field)
m

Map(center=[np.float64(40.44614584653234), np.float64(-94.82883374273663)], controls=(ZoomControl(options=['po…

## Save the selected gages to a local cache

Click a HUC boundary on the map above, then run this cell:

In [8]:
if selection['gages_df'] is not None:
    output_path = f"data/selected_{huc_field}_{selection['huc_id']}_gages.parquet"
    selection['gages_df'].to_parquet(output_path, index=False)
    print(f"Saved {len(selection['gages_df'])} gages to {output_path}")
else:
    print('No HUC selected yet — click a boundary on the map above first.')

Saved 1 gages to data/selected_huc8_07140102_gages.parquet
